In [ ]:
import numpy as np
import csv

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],    # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],    # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],    # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

# Task 1: Clean Machakos data with try/except ValueError
valid_vals = []
for val in machakos_raw:
    try:
        valid_vals.append(float(val.strip()))
    except ValueError:
        pass

machakos_avg = round(sum(valid_vals) / len(valid_vals))

machakos_cleaned = []
imputed_months = []

for i, val in enumerate(machakos_raw):
    try:
        num = float(val.strip())
        machakos_cleaned.append(int(num))
    except ValueError:
        machakos_cleaned.append(machakos_avg)
        imputed_months.append(months[i])

print(f"Machakos Imputed Average Value: {machakos_avg}")
print(f"Months imputed due to missing/invalid data: {imputed_months}")

Machakos Imputed Average Value: 68
Months imputed due to missing/invalid data: ['Apr', 'Sep']


In [ ]:
# Task 2: Combine all five counties into a 5 x 12 NumPy array
clean_rows.append(machakos_cleaned)
rainfall_matrix = np.array(clean_rows)

print("Rainfall Matrix Shape:", rainfall_matrix.shape)
print("Rainfall Matrix Dtype:", rainfall_matrix.dtype)

Rainfall Matrix Shape: (5, 12)
Rainfall Matrix Dtype: int64


In [ ]:
# Task 3: Annual total rainfall per county and average rainfall per month
annual_totals = rainfall_matrix.sum(axis=1)
monthly_averages = rainfall_matrix.mean(axis=0).round(1)

for c, total in zip(counties, annual_totals):
    print(f"{c} Annual Total: {total} mm")

print("\nMonthly Averages across all counties:", monthly_averages)

# Task 4: Wettest month per county using np.argmax with axis=1
wettest_month_indices = np.argmax(rainfall_matrix, axis=1)
wettest_months = [months[i] for i in wettest_month_indices]

for c, wm in zip(counties, wettest_months):
    print(f"{c} Wettest Month: {wm}")

# Task 5: Count dry months (below 50 mm) using boolean masks
dry_masks = rainfall_matrix < 50
dry_counts = dry_masks.sum(axis=1)

for c, dc in zip(counties, dry_counts):
    print(f"{c} Dry Months Count: {dc}")

# Task 6: Identify two consecutive months with highest combined rainfall (Planting Window)
two_month_pairs = rainfall_matrix[:, :-1] + rainfall_matrix[:, 1:]
best_pair_indices = two_month_pairs.argmax(axis=1)

planting_windows = []
for idx in best_pair_indices:
    window_str = f"{months[idx]}–{months[idx+1]}"
    planting_windows.append(window_str)

for c, pw in zip(counties, planting_windows):
    print(f"{c} Recommended Planting Window: {pw}")

Nairobi Annual Total: 1045 mm
Nakuru Annual Total: 916 mm
Kisumu Annual Total: 1371 mm
Mombasa Annual Total: 1075 mm
Machakos Annual Total: 811 mm

Monthly Averages across all counties: [ 47.   47.6  91.  164.4 171.2  67.6  56.6  58.2  64.2  67.8 127.4  80.6]
Nairobi Wettest Month: Apr
Nakuru Wettest Month: Apr
Kisumu Wettest Month: Apr
Mombasa Wettest Month: May
Machakos Wettest Month: Nov
Nairobi Dry Months Count: 4
Nakuru Dry Months Count: 3
Kisumu Dry Months Count: 0
Mombasa Dry Months Count: 2
Machakos Dry Months Count: 6
Nairobi Recommended Planting Window: Apr–May
Nakuru Recommended Planting Window: Apr–May
Kisumu Recommended Planting Window: Apr–May
Mombasa Recommended Planting Window: Apr–May
Machakos Recommended Planting Window: Nov–Dec


In [ ]:
# Task 7: Write rainfall_summary.csv using the csv module
csv_filename = "rainfall_summary.csv"
fields = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

summary_data = []
for i in range(len(counties)):
    summary_data.append({
        "county": counties[i],
        "annual_total_mm": int(annual_totals[i]),
        "wettest_month": wettest_months[i],
        "dry_months": int(dry_counts[i]),
        "planting_window": planting_windows[i]
    })

with open(csv_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(summary_data)

print(f"Successfully generated and saved {csv_filename}")

Successfully generated and saved rainfall_summary.csv


## Planting Advisory

Farmers across all five counties should closely align crop preparation with their respective two-month peak rainfall windows to maximize seed germination and early root development. In regions such as Kisumu and Nairobi, where seasonal rainfall totals are higher, early planting ahead of peak months ensures crops utilize maximum moisture. Conversely, counties like Machakos and Mombasa experience higher dry-month counts, necessitating drought-resistant crop varieties and localized water-harvesting techniques. Agricultural extension officers should prioritize SMS advisories right before the onset of the recommended planting windows.

## Data Quality Notes

The raw dataset presented notable inconsistencies, particularly in the Machakos weather station readings, which arrived as raw text containing text errors like "N/A" and blank strings. These anomalies were successfully intercepted and handled using specific exception handling (`ValueError`), allowing us to isolate valid numeric entries and compute a reliable station average (68 mm) for imputation. Stakeholders should note that while mathematical imputation allows complete matrix analysis, imputed values are estimates and should be verified against regional meteorological station logs before deploying large-scale agricultural interventions.